# Étape 2 : nettoyage des textes

Minuscules, suppression des emojis et de la ponctuation, espaces normalisés. Le `_` est conservé : les balises (`personne_tag`, `ip_tag`...) restent intactes.
Entrée : `data/anonymise/train_anonyme_2.csv`, `data/anonymise/test_anonyme_2.csv` — Sortie : `data/clean/train_clean.csv`, `data/clean/test_clean.csv`.

In [7]:
import re
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm   # barre de progression

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # racine du projet (notebook lancé depuis notebooks/ ou depuis la racine)

In [8]:
# Emojis et pictogrammes (plages Unicode courantes : symboles, dingbats, drapeaux, émojis récents)
RE_EMOJI = re.compile(
    "[\U0001F000-\U0001FAFF"   # émojis, pictogrammes, drapeaux
    "\U00002600-\U000027BF"    # symboles divers et dingbats
    "\U00002B00-\U00002BFF"    # flèches et symboles divers
    "\u200d\ufe0f]+",          # liant d'émojis et sélecteur de variation
    flags=re.UNICODE)


def clean_text(text: str) -> str:
    """Nettoie un commentaire déjà anonymisé."""
    text = text.lower()                          # 1. minuscules
    text = RE_EMOJI.sub(" ", text)               # 2. retire les emojis
    text = re.sub(r"[^\w\s]", " ", text)         # 3. retire la ponctuation ("_" conservé -> balises intactes)
    return re.sub(r"\s+", " ", text).strip()     # 4. espaces multiples et retours à la ligne -> un espace

In [9]:
# --- Application à la colonne comment_text + vérification des balises ---
# Les balises ont la forme xxx_tag (personne_tag, ip_tag, url_tag...)
RE_BALISE = r"(?i)[a-z]+_tag(?![a-z0-9_])"   # insensible à la casse ; ne compte pas "_tagging" etc.

DOSSIER_CLEAN = RACINE / "data" / "clean"           # dossier dédié aux textes nettoyés
DOSSIER_CLEAN.mkdir(parents=True, exist_ok=True)
FICHIERS_CLEAN = {
    RACINE / "data" / "anonymise" / "train_anonyme_2.csv": DOSSIER_CLEAN / "train_clean.csv",
    RACINE / "data" / "anonymise" / "test_anonyme_2.csv": DOSSIER_CLEAN / "test_clean.csv",
}

for entree, sortie in FICHIERS_CLEAN.items():
    df = pd.read_csv(entree)
    avant = df["comment_text"].fillna("")
    apres = pd.Series([clean_text(t) for t in tqdm(avant, desc=entree.name)], index=avant.index)

    df["comment_text"] = apres
    df.to_csv(sortie, index=False)
    print(f"\n===== {sortie} ({len(df)} lignes) =====")

    # 5 exemples avant/après (de préférence des commentaires contenant une balise)
    avec_balise = avant[avant.str.contains("_tag")]
    for i in (avec_balise if len(avec_balise) >= 5 else avant).sample(5, random_state=0).index:
        print("AVANT :", avant[i][:200].replace("\n", " "))
        print("APRES :", apres[i][:200], "\n")

    # Vérification : le nombre de balises *_tag doit être identique, ligne par ligne, avant et après
    assert (avant.str.count(RE_BALISE) == apres.str.count(RE_BALISE)).all(), "une balise *_tag a été cassée"
    print("Vérification OK : aucune balise *_tag n'a été cassée.")

train_anonyme_2.csv: 100%|██████████| 159571/159571 [00:02<00:00, 56289.62it/s]



===== /Users/situn/Situu/Projects/depl_ai_cyber/data/clean/train_clean.csv (159571 lignes) =====
AVANT : ":You didn't ask for ""China specialists"", you asked for academics. There's a difference between the two. And in any case you seemed to know what to find, so why play a game and ask me to provide you
APRES : you didn t ask for china specialists you asked for academics there s a difference between the two and in any case you seemed to know what to find so why play a game and ask me to provide you with a li 

AVANT : ", 6 August 2015 (organisation_tag)   So ""consensus"" for the wrong template, uniquely applied to this article, consists of one guy defending its inclusion and another attacking it? I think organisat
APRES : 6 august 2015 organisation_tag so consensus for the wrong template uniquely applied to this article consists of one guy defending its inclusion and another attacking it i think organisation_tag s resi 

AVANT : The add {{fact}}, {{OR}}, and {{synthesis}} tags to th

test_anonyme_2.csv: 100%|██████████| 153164/153164 [00:02<00:00, 63619.10it/s]



===== /Users/situn/Situu/Projects/depl_ai_cyber/data/clean/test_clean.csv (153164 lignes) =====
AVANT : " one of the biggest RV makers in the US; has over $400M in revenues, and is traded on the organisation_tag. True, the article you speedied was a sub-stub, but maybe we should give it a chance to grow
APRES : one of the biggest rv makers in the us has over 400m in revenues and is traded on the organisation_tag true the article you speedied was a sub stub but maybe we should give it a chance to grow do you  

AVANT : Hmm. Sounds like you have no intention to leave Rawat articles for others then. You intend to refrain from editing these articles 'directly for now' but you still intend to report people whose behavio
APRES : hmm sounds like you have no intention to leave rawat articles for others then you intend to refrain from editing these articles directly for now but you still intend to report people whose behaviour y 

AVANT : == So this here is how it's gonna go ==    Hey there!  